# House Price Prediction: run one step at a time
Run each cell from top to bottom. You can inspect the dataset and plots before running the model cells.


## 1. Imports and paths

In [ ]:
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT = Path.cwd()
if not (PROJECT / "AmesHousing.csv").is_file():
    candidate = PROJECT / "OIBSIP-DataAnalytics-L2-PredictingHousePricesWithLinearRegression-main"
    if (candidate / "AmesHousing.csv").is_file():
        PROJECT = candidate
DATA_FILE = PROJECT / "AmesHousing.csv"
OUTPUT_DIR = PROJECT / "outputs" / ("notebook_run_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load and inspect Ames Housing

In [ ]:
data = pd.read_csv(DATA_FILE, low_memory=False)
if "SalePrice" not in data.columns:
    raise ValueError("AmesHousing.csv must contain SalePrice")
print("Rows and columns:", data.shape)
print("Duplicate rows:", data.duplicated().sum())
display(data.head())
display(data.isna().sum().sort_values(ascending=False).head(15))
display(data["SalePrice"].describe())


## 3. Clean target and create HouseAge

In [ ]:
data["SalePrice"] = pd.to_numeric(data["SalePrice"], errors="coerce")
data = data[data["SalePrice"].notna() & (data["SalePrice"] > 0)].drop_duplicates().copy()
year_built = next((c for c in data.columns if c.lower().replace(" ", "") in {"yearbuilt"}), None)
year_sold = next((c for c in data.columns if c.lower().replace(" ", "") in {"yrsold", "yearsold"}), None)
if year_built and year_sold:
    data["HouseAge"] = pd.to_numeric(data[year_sold], errors="coerce") - pd.to_numeric(data[year_built], errors="coerce")
print("Rows after target cleaning:", len(data), "HouseAge created:", bool(year_built and year_sold))


## 4. Explore price and feature relationships

In [ ]:
sns.histplot(data["SalePrice"], bins=40, kde=True)
plt.title("Sale price distribution")
plt.show()
living_area = next((c for c in data.columns if c.lower().replace(" ", "") in {"grlivarea"}), None)
if living_area:
    sns.scatterplot(data=data, x=living_area, y="SalePrice", alpha=.5)
    plt.title("Living area vs sale price")
    plt.show()
quality = next((c for c in data.columns if c.lower().replace(" ", "") in {"overallqual"}), None)
if quality:
    sns.barplot(data=data.groupby(quality, as_index=False)["SalePrice"].mean(), x=quality, y="SalePrice")
    plt.title("Average sale price by overall quality")
    plt.show()
display(data.describe(include="all").T.head(20))


## 5. Split data and build preprocessing

In [ ]:
target = data["SalePrice"]
X = data.drop(columns=["SalePrice", "Order", "PID"], errors="ignore")
numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.columns.difference(numeric_features).tolist()
X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=.2, random_state=42)

def make_pipeline(estimator):
    prep = ColumnTransformer([
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                              ("scale", StandardScaler())]), numeric_features),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                                  ("encode", OneHotEncoder(handle_unknown="ignore", min_frequency=5))]),
         categorical_features),
    ])
    return Pipeline([("preprocess", prep), ("model", estimator)])
print("Training rows:", len(X_train), "Test rows:", len(X_test))


## 6. Fit and compare Linear, Ridge, and Lasso

In [ ]:
estimators = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10),
    "Lasso Regression": Lasso(alpha=100, max_iter=20000),
}
fitted_models, predictions, metric_rows = {}, {}, []
for name, estimator in estimators.items():
    model = make_pipeline(estimator)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted_models[name] = model
    predictions[name] = pred
    metric_rows.append({"model": name, "mse": mean_squared_error(y_test, pred),
        "rmse": np.sqrt(mean_squared_error(y_test, pred)),
        "mae": mean_absolute_error(y_test, pred), "r2": r2_score(y_test, pred)})
model_comparison = pd.DataFrame(metric_rows).sort_values("rmse")
display(model_comparison)
best_model_name = model_comparison.iloc[0]["model"]
best_predictions = predictions[best_model_name]


## 7. Inspect residuals and coefficients

In [ ]:
residuals = y_test.to_numpy() - best_predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, best_predictions, alpha=.6)
lo, hi = min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())
axes[0].plot([lo, hi], [lo, hi], "r--")
axes[0].set(xlabel="Actual", ylabel="Predicted", title=f"Actual vs predicted: {best_model_name}")
axes[1].scatter(best_predictions, residuals, alpha=.6)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set(xlabel="Predicted", ylabel="Residual", title="Residual vs predicted")
plt.tight_layout()
plt.show()
linear = fitted_models["Linear Regression"]
feature_names = linear.named_steps["preprocess"].get_feature_names_out()
coefficients = pd.DataFrame({"feature": feature_names,
    "coefficient": linear.named_steps["model"].coef_})
display(coefficients.reindex(coefficients.coefficient.abs().sort_values(ascending=False).index).head(20))


## 8. Save README deliverables

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"actual_sale_price": y_test.to_numpy(), "best_model": best_model_name,
    "predicted_sale_price": best_predictions, "residual": residuals}).to_csv(
    OUTPUT_DIR / "house_price_predictions.csv", index=False)
coefficients.to_csv(OUTPUT_DIR / "linear_regression_coefficients.csv", index=False)
model_comparison.to_csv(OUTPUT_DIR / "house_price_model_comparison.csv", index=False)
print("Saved outputs to:", OUTPUT_DIR)

# Export the required metrics, missing-value analysis, coefficients and plots.
model_comparison.to_csv(OUTPUT_DIR / "house_price_model_comparison.csv", index=False)
coefficients.to_csv(OUTPUT_DIR / "linear_regression_coefficients.csv", index=False)
pd.DataFrame({"actual_sale_price": y_test.to_numpy(), "best_model": best_model_name,
    "predicted_sale_price": best_predictions, "residual": residuals,
    "absolute_error": np.abs(residuals)}).to_csv(OUTPUT_DIR / "house_price_predictions.csv", index=False)
data.isna().sum().rename("missing_values").to_frame().to_csv(OUTPUT_DIR / "missing_value_summary.csv")
data.describe(include="all").T.to_csv(OUTPUT_DIR / "eda_descriptive_statistics.csv")
fig, ax = plt.subplots(figsize=(7, 6)); ax.scatter(y_test, best_predictions, alpha=.6)
lo, hi = min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())
ax.plot([lo, hi], [lo, hi], "r--"); ax.set(xlabel="Actual", ylabel="Predicted", title=f"Actual vs predicted: {best_model_name}")
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "actual_vs_predicted.png", dpi=150); plt.close(fig)
fig, axes = plt.subplots(1, 2, figsize=(11, 4)); axes[0].scatter(best_predictions, residuals, alpha=.6); axes[0].axhline(0, color="red", ls="--")
axes[0].set(title="Residual vs predicted", xlabel="Predicted", ylabel="Residual"); axes[1].hist(residuals, bins=35); axes[1].set(title="Residual distribution", xlabel="Residual")
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "residual_analysis.png", dpi=150); plt.close(fig)
